> **Course source:** [llama-index/agents.ipynb](https://huggingface.co/agents-course/notebooks/blob/cecf8edffb02101c90634b84076bd93b1a20c713/unit2/llama-index/agents.ipynb).
> **Compatibility:** Reuses the course Chroma collection. Explicit ReActAgent examples use FunctionAgent with Groq's native function calling. The agent runs verify the arithmetic and retrieval tool calls.

# Agents in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
COURSE_PACKAGES = ['llama-index-core==0.14.26', 'llama-index-llms-openai-like==0.7.2', 'llama-index-embeddings-huggingface==0.9.0', 'llama-index-vector-stores-chroma==0.7.0', 'datasets==5.2.0']

import os
import subprocess
import sys

if not os.environ.get("HF_COURSE_BATCH"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--disable-pip-version-check", *COURSE_PACKAGES], check=True, timeout=600)

And, let's log in to Hugging Face to use serverless Inference APIs.

In [2]:
import json
import os
from functools import lru_cache

import httpx
from openai import OpenAI

GROQ_BASE = "https://api.groq.com/openai/v1"
QWEN = "qwen/qwen3.8-27b"
GPT_OSS = ("openai/gpt-oss-20b", "openai/gpt-oss-120b")


def optional_secret(name):
    """Read a notebook-authorized secret without printing it."""
    if os.getenv(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return None
    except Exception as exc:
        if type(exc).__name__ in {"SecretNotFoundError", "NotebookAccessError"}:
            return None
        raise


@lru_cache(maxsize=1)
def groq_account():
    key = optional_secret("GROQ_API_KEY")
    if not key:
        raise RuntimeError("Enable notebook access to the GROQ_API_KEY secret in Colab.")
    with OpenAI(api_key=key, base_url=GROQ_BASE, timeout=20, max_retries=0) as client:
        return key, frozenset(model.id for model in client.models.list().data)


class ModelRouting:
    def __init__(self, models, mode):
        self.models = list(models)
        self.mode = mode

    def requests(self, request):
        if request.method != "POST" or not request.url.path.endswith("/chat/completions"):
            yield None, request
            return
        body = json.loads(request.content)
        headers = {key: val for key, val in request.headers.items()
                   if key not in {"content-length", "transfer-encoding"}}
        if self.mode == "code" and body.get("tools"):
            raise ValueError("CodeAgent uses Python actions. Its API request must have no native tools.")
        for model_id in tuple(self.models):
            payload = {**body, "model": model_id}
            if not payload.get("tools"):
                payload.pop("tools", None)
                payload.pop("tool_choice", None)
                payload.pop("parallel_tool_calls", None)
            max_tokens = payload.pop("max_tokens", None)
            if max_tokens and "max_completion_tokens" not in payload:
                payload["max_completion_tokens"] = max_tokens
            if model_id in GPT_OSS:
                payload.pop("parallel_tool_calls", None)
                payload.pop("stop", None)
                payload["reasoning_effort"] = "low"
            elif model_id == QWEN:
                payload["reasoning_effort"] = "none"
            yield model_id, httpx.Request(
                request.method, request.url, json=payload, headers=headers,
                extensions=request.extensions,
            )

    def accept(self, model_id, status, attempt):
        if model_id is None:
            return True
        if status in {429, 503} and attempt < len(self.models):
            return False
        if 200 <= status < 300 and model_id != self.models[0]:
            self.models.remove(model_id)
            self.models.insert(0, model_id)
            print(f"Groq fallback model: {model_id}", flush=True)
        return True


class CourseTransport(httpx.HTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    def handle_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = super().handle_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            response.close()


class AsyncCourseTransport(httpx.AsyncHTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    async def handle_async_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = await super().handle_async_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            await response.aclose()


@lru_cache(maxsize=3)
def groq_route(mode):
    key, available = groq_account()
    candidates = {
        "text": (QWEN, *GPT_OSS),
        "vision": (QWEN,),
        "code": (QWEN, *GPT_OSS),
    }[mode]
    models = [model_id for model_id in candidates if model_id in available]
    if not models:
        raise RuntimeError(f"None of the Groq {mode} models are enabled for this project: {candidates}")
    print(f"Groq {mode} candidates, in order: {', '.join(models)}", flush=True)
    return key, ModelRouting(models, mode)


@lru_cache(maxsize=1)
def make_groq_llm():
    from llama_index.llms.openai_like import OpenAILike
    key, routing = groq_route("text")
    return OpenAILike(
        model=routing.models[0], api_base=GROQ_BASE, api_key=key,
        is_chat_model=True, is_function_calling_model=True, context_window=32768,
        max_tokens=1024, temperature=0.1, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        async_http_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=2)
def make_chat_model(mode="text"):
    from langchain_openai import ChatOpenAI
    key, routing = groq_route(mode)
    return ChatOpenAI(
        model=routing.models[0], base_url=GROQ_BASE, api_key=key,
        temperature=0.1, max_tokens=1024, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        http_async_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=1)
def make_code_model():
    from smolagents import OpenAIModel
    key, routing = groq_route("code")
    return OpenAIModel(
        model_id=routing.models[0], api_base=GROQ_BASE, api_key=key,
        max_tokens=1800, temperature=0.1, retry=False,
        client_kwargs={
            "http_client": httpx.Client(transport=CourseTransport(routing), timeout=30),
            "timeout": 30, "max_retries": 0,
        },
    )
from pathlib import Path
from hashlib import sha256

from datasets import load_dataset
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

PERSONAS = "dvilasuero/finepersonas-v0.1-tiny"


@lru_cache(maxsize=1)
def persona_embedding():
    return HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", device="cpu")


def persona_documents(folder, only_first_ten=False):
    dataset = load_dataset(PERSONAS, split="train")
    folder.mkdir(parents=True, exist_ok=True)
    indices = sorted(range(len(dataset)), key=lambda i: f"persona_{i}.txt")
    if only_first_ten:
        indices = indices[:10]
    files = []
    for i in indices:
        path = folder / f"persona_{i}.txt"
        path.write_text(dataset[i]["persona"], encoding="utf-8")
        files.append(str(path))
    return SimpleDirectoryReader(input_files=files, encoding="utf-8").load_data()


async def ingest_personas(collection, vector_store, documents):
    nodes = await IngestionPipeline(
        transformations=[SentenceSplitter()], 
    ).arun(documents=documents[:10])
    for node in nodes:
        node.id_ = sha256((node.metadata["file_name"] + "\n" + node.text).encode()).hexdigest()
    existing = set(collection.get(include=[])["ids"])
    missing = [node for node in nodes if node.node_id not in existing]
    if missing:
        new_nodes = await IngestionPipeline(
            transformations=[persona_embedding()], vector_store=vector_store,
        ).arun(nodes=missing)
        embeddings = {node.node_id: node.embedding for node in new_nodes}
    else:
        embeddings = {}
    if len(embeddings) != len(missing) or not collection.count():
        raise RuntimeError("The Chroma ingestion did not persist all expected persona nodes.")
    return nodes

## Initialising agents

Let's start by initialising an agent. We will use the basic `AgentWorkflow` class to create an agent.

In [3]:
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = make_groq_llm()

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    timeout=120,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools. For arithmetic, always call the relevant tools before giving the result.",
)

Groq text candidates, in order: qwen/qwen3.8-27b, openai/gpt-oss-20b, openai/gpt-oss-120b


Then, we can run the agent and get the response and reasoning behind the tool calls.

In [4]:
successful_tools = set()
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        if not ev.tool_output.is_error:
            successful_tools.add(ev.tool_name)
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
assert {"add", "multiply"} <= successful_tools, sorted(successful_tools)
resp

Groq fallback model: openai/gpt-oss-20b

Called tool:  add {'a': 2, 'b': 2} => 4

Called tool:  multiply {'a': 4, 'b': 2} => 8
The result is **8**.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='thinking', content='We need to output final answer.', num_tokens=None, additional_information={}), TextBlock(block_type='text', text='The result is **8**.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-c4a0be95-4d86-49aa-8049-b41b5c161319', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'model': 'openai/gpt-oss-20b', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': None, 'system_fingerprint': 'fp_e11c8bbf69', 'usage': {'completion_tokens': 24, 'prompt_tokens': 391, 'total_tokens': 415, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 8, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.20

In a similar fashion, we can pass state and context to the agent.


In [5]:
from llama_index.core.workflow import Context

memory_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[], llm=llm,
    system_prompt="You are a conversational assistant. Remember details the user shares and recall them when asked.",
    timeout=100,
)
ctx = Context(memory_agent)

first_turn = memory_agent.run("My name is Bob.", ctx=ctx)
await first_turn
completed_ctx = first_turn.ctx
conversation_memory = await completed_ctx.store.get("memory")
recorded_history = await conversation_memory.aget()
assert any("My name is Bob." in (message.content or "") for message in recorded_history), recorded_history
response = await memory_agent.run("What was my name again?", ctx=completed_ctx, chat_history=recorded_history)
response
print("Memory response:", response)
assert "bob" in str(response).lower(), str(response)
response

Memory response: Your name is Bob.


/usr/local/lib/python3.13/dist-packages/workflows/context/state_store.py:231: UnserializableKeyWarning: Skipping serialization of known unserializable key: memory -- This is expected but will require this item to be set manually after deserialization.
  warnings.warn(


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='thinking', content='Recall Bob.', num_tokens=None, additional_information={}), TextBlock(block_type='text', text='Your name is Bob.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-80fdc6bf-4d49-402b-aefc-fc98e8c42106', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'model': 'openai/gpt-oss-20b', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': None, 'system_fingerprint': 'fp_816fc87d18', 'usage': {'completion_tokens': 18, 'prompt_tokens': 126, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 4, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.210797401, 'prompt_time':

## Creating RAG Agents with QueryEngineTools

Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. We will pass it to the `AgentWorkflow` class to create a RAG agent.

In [6]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create a query engine
embed_model = persona_embedding()

if not chroma_collection.count():
    seed_docs = persona_documents(Path("data"), only_first_ten=True)
    await ingest_personas(chroma_collection, vector_store, seed_docs)
print("Indexed persona chunks:", chroma_collection.count())
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)

Indexed persona chunks: 10


Loading weights: 100%|##########| 199/199 [00:00<00:00, 3668.46it/s]


In [7]:
llm = make_groq_llm()
query_engine = index.as_query_engine(llm=llm)
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant. Call the personas tool to search the database before answering persona questions.",
)

And, we can once more get the response and reasoning behind the tool calls.

In [8]:
successful_tools = set()
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions."
)
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        if not ev.tool_output.is_error:
            successful_tools.add(ev.tool_name)
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
assert {"personas"} <= successful_tools, sorted(successful_tools)
resp


Called tool:  personas {'input': 'science fiction'} => Science fiction is a genre that explores imaginative concepts such as advanced technology, space travel, time manipulation, and alternate realities. It often uses speculative settings to examine human nature, societal structures, and ethical dilemmas, blending scientific possibilities with creative storytelling.
**Science Fiction Persona Descriptions**

- *Science fiction is a genre that explores imaginative concepts such as advanced technology, space travel, time manipulation, and alternate realities. It often uses speculative settings to examine human nature, societal structures, and ethical dilemmas, blending scientific possibilities with creative storytelling.*

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='thinking', content='We need to return persona descriptions. The tool returned a description. Probably just output that.', num_tokens=None, additional_information={}), TextBlock(block_type='text', text='**Science Fiction Persona Descriptions**\n\n- *Science fiction is a genre that explores imaginative concepts such as advanced technology, space travel, time manipulation, and alternate realities. It often uses speculative settings to examine human nature, societal structures, and ethical dilemmas, blending scientific possibilities with creative storytelling.*')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-a1a4884a-1630-48eb-9649-72638c2e780b', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'model': 'ope

## Creating multi-agent systems

We can also create multi-agent systems by passing multiple agents to the `AgentWorkflow` class.

In [9]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
)


# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = FunctionAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = FunctionAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm,
)

# Create and run the workflow
agent = AgentWorkflow(agents=[calculator_agent, query_agent], root_agent="calculator")

# The next cell starts the workflow inside its active event loop.

In [10]:
handler = agent.run(user_msg="Can you add 5 and 3?")
successful_tools = set()
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        if not ev.tool_output.is_error:
            successful_tools.add(ev.tool_name)
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
assert {"add"} <= successful_tools, sorted(successful_tools)
resp


Called tool:  add {'a': 5, 'b': 3} => 8
The sum of 5 and 3 is **8**.

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='thinking', content='We have answer.', num_tokens=None, additional_information={}), TextBlock(block_type='text', text='The sum of 5 and 3 is **8**.')]), structured_response=None, current_agent_name='calculator', raw={'id': 'chatcmpl-71a53364-721b-40e7-bae3-1f71e9040362', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'model': 'openai/gpt-oss-20b', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': None, 'system_fingerprint': 'fp_d3e146e1a5', 'usage': {'completion_tokens': 27, 'prompt_tokens': 336, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 5, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.01855